# Fly Locomotion Analysis

Load SLEAP `.slp` files, extract locomotion features, and generate all figures.

**Quick start**
1. Edit the `GROUPS` dict in the *Configuration* cell with your file paths.
2. Run all cells top-to-bottom (`Kernel → Restart & Run All`).
3. Figures are saved to `figures/` as PDFs and displayed inline.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sleap_io as sio
from scipy.optimize import curve_fit
from scipy.ndimage import gaussian_filter1d
from scipy.stats import pearsonr, spearmanr

# If fla is not installed, add the repo src/ to path
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from fla import (
    get_bodypart_coords, get_position_diff,
    get_frames_when_walking_forward, get_average_speed_for_ranges,
    get_raw_instantaneous_speeds, get_all_angles,
    get_average_gait_index_for_ranges, get_instantaneous_speeds_and_stance,
    get_gait_probability_distribution, convert_ranges_to_binary,
    get_swing_durations, get_stance_durations,
    get_step_lengths, get_period_durations, get_footprint_alignment,
    plot_swing_stance, linear, power_law,
    FPS,
)

os.makedirs("figures", exist_ok=True)

## Configuration

Edit `GROUPS` to point to your `.slp` files.  Each entry under `"files"` can be:

| Format | Loads |
|--------|-------|
| `"path/to/file.slp"` | every video × every track in the file |
| `("path/to/file.slp", video_idx)` | all tracks from one video |
| `("path/to/file.slp", video_idx, track_idx)` | a single track from one video |

Mix formats freely within a group.

In [ ]:
# ── EDIT THIS CELL ─────────────────────────────────────────────────────────────
GROUPS = {
    "Old Males": {
        "color": "#1750AC",
        "files": [
            "../slps/old_males.slp",           # all videos × all tracks
        ],
    },
    "Young Males": {
        "color": "#86CEFA",
        "files": [
            "../slps/young_males.slp",
        ],
    },
    # "Old Females": {
    #     "color": "#C63287",
    #     "files": [
    #         "path/to/IM061.slp",
    #     ],
    # },
    # "Young Females": {
    #     "color": "#FF99DA",
    #     "files": [
    #         "path/to/IM095.slp",
    #         # fine-grained examples:
    #         # ("path/to/IM096.slp", 1),      # video 1 only, all tracks
    #         # ("path/to/IM097.slp", 0, 2),   # video 0, track 2 only
    #     ],
    # },
}

ALPHA = 0.5          # scatter plot opacity
FIG_SIZE = (4, 3)    # default figure size (inches)
NUM_GROUPS = len(GROUPS)

## Load Data

In [ ]:
def _load_flies(cfg_files):
    """Expand flexible config entries into a list of fly dicts."""
    flies = []
    label_cache = {}
    for entry in cfg_files:
        if isinstance(entry, str):
            path, video_sel, track_sel = entry, None, None
        elif len(entry) == 2:
            path, video_sel, track_sel = entry[0], [entry[1]], None
        else:
            path, video_sel, track_sel = entry[0], [entry[1]], [entry[2]]

        if path not in label_cache:
            label_cache[path] = sio.load_slp(path)
        labels = label_cache[path]

        if video_sel is None:
            video_sel = range(len(labels.videos))

        for v in video_sel:
            if track_sel is None:
                trx = labels.numpy(video=v)
                n_valid = int((~np.all(np.isnan(trx), axis=(0, 2, 3))).sum())
                _track_sel = range(n_valid)
            else:
                _track_sel = track_sel
            for t in _track_sel:
                flies.append(get_bodypart_coords(labels, t, video=v, smoothing=False))
    return flies

fly_data = {}
for group, cfg in GROUPS.items():
    fly_data[group] = _load_flies(cfg["files"])
    print(f"{group}: {len(fly_data[group])} flies loaded")

## Feature Extraction

Run once — all downstream figure cells read from `group_data`.

In [ ]:
features = {}
for group, flies in fly_data.items():
    group_feats = []
    for i, fly in enumerate(flies):
        pos_diffs    = get_position_diff(fly, smoothing=True, sigma=1)
        fwf          = get_frames_when_walking_forward(fly, frame_req=20, units="mm", sigma=1, max_gap=1)
        # with open(f"{i}_fwf.pkl", "wb") as file:
        #     pickle.dump(fwf, file)
        binary_ranges = convert_ranges_to_binary(fly, fwf)

        fp_R, fp_L = get_footprint_alignment(fly, fwf, pos_diffs)

        group_feats.append({
            "fly":           fly,
            "fwf":           fwf,
            "pos_diffs":     pos_diffs,
            "binary_ranges": binary_ranges,
            "avg_speeds":    get_average_speed_for_ranges(fly, fwf, units="mm"),
            "inst_speeds":   get_raw_instantaneous_speeds(fly, units="mm"),
            "angles":        get_all_angles(fly, fwf),
            "gait_index":    get_average_gait_index_for_ranges(fly, fwf),
            "sal":           get_instantaneous_speeds_and_stance(fly, units="mm", stance_type="leg", mm_limit=20),
            "sag":           get_instantaneous_speeds_and_stance(fly, units="mm", stance_type="gait", mm_limit=20),
            "swing_dur":     get_swing_durations(binary_ranges),
            "stance_dur":    get_stance_durations(binary_ranges),
            "step_len":      get_step_lengths(fly, fwf, pos_diffs, units="mm"),
            "periods":       get_period_durations(fwf, pos_diffs),
            "fp_R":          fp_R,
            "fp_L":          fp_L,
        })
    features[group] = group_feats
    print(f"{group}: features extracted")

In [ ]:
# Aggregate across flies within each group for convenient plotting
group_data = {}

for group, feats in features.items():
    avg_speeds  = np.concatenate([f["avg_speeds"]  for f in feats])
    inst_speeds = np.concatenate([f["inst_speeds"] for f in feats])
    angles      = np.array([a for f in feats for a in f["angles"]])
    gait_index  = np.concatenate([f["gait_index"] for f in feats])
    sal_list    = [f["sal"] for f in feats if len(f["sal"]) > 0]
    sal         = np.concatenate(sal_list) if sal_list else np.empty((0, 2))
    sag_list    = [f["sag"] for f in feats if len(f["sag"]) > 0]
    sag         = np.concatenate(sag_list) if sag_list else np.empty((0, 2))

    # Per-bout mean durations paired with per-bout average speeds
    def _paired(speed_list, dur_list):
        sp, mn = [], []
        for spd, durs in zip(speed_list, dur_list):
            if durs:
                sp.append(spd); mn.append(np.mean(durs))
        return np.array(sp), np.array(mn)

    sw_sp, sw_mn = zip(*[_paired(f["avg_speeds"], f["swing_dur"])  for f in feats]) if feats else ([], [])
    st_sp, st_mn = zip(*[_paired(f["avg_speeds"], f["stance_dur"]) for f in feats]) if feats else ([], [])

    swing_speeds  = np.concatenate([s for s in sw_sp if len(s)])
    swing_means   = np.concatenate([m for m in sw_mn if len(m)])
    stance_speeds = np.concatenate([s for s in st_sp if len(s)])
    stance_means  = np.concatenate([m for m in st_mn if len(m)])

    # Periods and step lengths (drop NaNs)
    per_sp, per_mn = [], []
    sl_sp,  sl_mn  = [], []
    fp_sp,  fp_mn  = [], []
    bout_lengths   = []

    for f in feats:
        for spd, per in zip(f["avg_speeds"], f["periods"]):
            if not np.isnan(per):
                per_sp.append(spd); per_mn.append(per)
        for spd, sl in zip(f["avg_speeds"], f["step_len"]):
            if not np.isnan(sl):
                sl_sp.append(spd); sl_mn.append(sl)
        for spd, r, l in zip(f["avg_speeds"], f["fp_R"], f["fp_L"]):
            if not (np.isnan(r) or np.isnan(l)):
                fp_sp.append(spd); fp_mn.append((r + l) / 2)
        for start, end in f["fwf"]:
            bout_lengths.append(end - start)

    group_data[group] = dict(
        avg_speeds    = avg_speeds,
        inst_speeds   = inst_speeds,
        angles        = angles,
        gait_index    = gait_index,
        sal           = np.array(sal),
        sag           = np.array(sag),
        swing_speeds  = np.array(swing_speeds),
        swing_means   = np.array(swing_means),
        stance_speeds = np.array(stance_speeds),
        stance_means  = np.array(stance_means),
        period_speeds = np.array(per_sp),
        period_means  = np.array(per_mn),
        steplen_speeds= np.array(sl_sp),
        steplen_means = np.array(sl_mn),
        fp_speeds     = np.array(fp_sp),
        fp_means      = np.array(fp_mn),
        bout_lengths  = np.array(bout_lengths) / FPS,  # convert to seconds
    )

print("Aggregation complete.")

## Speed Analysis

In [ ]:
# Average speed distribution
fig, ax = plt.subplots(figsize=FIG_SIZE)
for group, data in group_data.items():
    speeds = data["avg_speeds"]
    hist, edges = np.histogram(speeds, bins=25, density=True)
    hist = gaussian_filter1d(hist.astype(float), sigma=0.8)
    centers = (edges[:-1] + edges[1:]) / 2
    ax.plot(centers, hist, color=GROUPS[group]["color"], label=group)

ax.set_xlabel("Average speed (mm/s)")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
# fig.savefig("figures/avg_speed_distribution.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Instantaneous speed distributions (per group, truncated at speed_limit)
speed_limit = 35  # mm/s

fig, axs = plt.subplots(NUM_GROUPS, figsize=(4, 3 * NUM_GROUPS))
for ax, (group, data) in zip(axs, group_data.items()):
    speeds = data["inst_speeds"]
    speeds_trunc = speeds[speeds <= speed_limit]
    ax.hist(speeds_trunc, bins=50, color=GROUPS[group]["color"],
            alpha=ALPHA, density=False)
    ax.set_title(group)
    ax.set_xlabel("Instantaneous Speed (mm/s)")
    ax.set_ylabel("Count")

fig.tight_layout()
# fig.savefig("figures/inst_speed_distribution.pdf", bbox_inches="tight")
plt.show()

## Turning Angle Analysis

In [ ]:
angle_limit = 120  # degrees

fig, ax = plt.subplots(figsize=FIG_SIZE)
for group, data in group_data.items():
    angles = data["angles"]
    angles_trunc = angles[np.abs(angles) < angle_limit]
    hist, edges = np.histogram(angles_trunc, bins=30, density=True)
    hist = gaussian_filter1d(hist.astype(float), sigma=0.5)
    centers = (edges[:-1] + edges[1:]) / 2
    ax.plot(centers, hist, color=GROUPS[group]["color"], label=group)

ax.set_xlabel("Turning angle (°)")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
# fig.savefig("figures/turning_angles.pdf", bbox_inches="tight")
plt.show()

## Gait Analysis

In [ ]:
# Gait index vs average speed
y_ticks = [-1.0, -0.5, 0, 0.5, 1.0]

fig, axs = plt.subplots(NUM_GROUPS, figsize=(4, 3 * NUM_GROUPS))
for ax, (group, data) in zip(axs.flatten(), group_data.items()):
    speeds, gi = data["avg_speeds"], data["gait_index"]
    valid = np.isfinite(gi)
    speeds, gi = speeds[valid], gi[valid]
    x_fit = np.linspace(speeds.min(), speeds.max(), 200)
    color  = GROUPS[group]["color"]
    ax.scatter(speeds, gi, color=color, alpha=ALPHA, s=8, label=group)
    try:
        params, _ = curve_fit(linear, speeds, gi)
        ax.plot(x_fit, linear(x_fit, *params), color=color)
        ax.set_yticks(y_ticks)
        r, _ = pearsonr(speeds, gi)
        ax.text(0.95, 0.50, f"Pearson r = {r:.2f}", transform=ax.transAxes,
                ha="right", va="top", fontsize=8)
    except RuntimeError:
        pass

ax.axhline(0, color="gray", lw=0.5, ls="--")
ax.set_xlabel("Average speed (mm/s)")
ax.set_ylabel("Gait index")
ax.legend()
fig.tight_layout()
# fig.savefig("figures/gait_index_vs_speed.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# P(# legs in swing)
fig, axs = plt.subplots(NUM_GROUPS, figsize=(6, 3.5 * NUM_GROUPS))
axs = np.atleast_1d(axs)

for ax, (group, data) in zip(axs, group_data.items()):
    sal = data["sal"]
    if len(sal) < 2:
        continue
    bin_centers, pct = get_gait_probability_distribution(sal, bins=30, stance_type="leg")
    for num_legs in range(7):
        smooth_pct = gaussian_filter1d(pct[num_legs], sigma=1.00)
        ax.plot(bin_centers, smooth_pct, label=f"{num_legs} legs in swing")
    ax.set_title(group)
    ax.set_xlabel("Instantaneous speed (mm/s)")
    ax.set_ylabel("Probability")
    ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# P(tetrapod) and P(tripod)
fig, axs = plt.subplots(NUM_GROUPS, figsize=(6, 3.5 * NUM_GROUPS))
axs = np.atleast_1d(axs)

for ax, (group, data) in zip(axs, group_data.items()):
    sag = data["sag"]
    if len(sal) < 2:
        continue
    bin_centers, pct = get_gait_probability_distribution(sag, bins=30, stance_type="gait")
    for gait in ["tetrapod", "noncanonical", "tripod"]:
        smooth_pct = gaussian_filter1d(pct[gait], sigma=1.00)
        ax.plot(bin_centers, smooth_pct, label=gait)
    ax.set_title(group)
    ax.set_xlabel("Instantaneous speed (mm/s)")
    ax.set_ylabel("Probability")
    ax.legend()

fig.tight_layout()
plt.show()

## Step Duration Analysis

In [ ]:
# Swing and stance duration vs speed
fig, axs = plt.subplots(NUM_GROUPS, 2, figsize=(8, 3.5 * NUM_GROUPS))
axs_flat = axs.flatten()

for i, (group, data) in enumerate(group_data.items()):
    color = GROUPS[group]["color"]
    ax_stance = axs_flat[i * 2]
    ax_swing  = axs_flat[i * 2 + 1]

    for ax, sp_key, mn_key, ylabel, r_coef, fit_fn, p0, r_label in [
        (ax_stance, "stance_speeds", "stance_means", "Stance duration (ms)", spearmanr, power_law, [100, -0.5], "Spearman r"),
        (ax_swing,  "swing_speeds",  "swing_means",  "Swing duration (ms)",  pearsonr,  linear,    [1, 0],      "Pearson r"),
    ]:
        sp, mn = data[sp_key], data[mn_key]
        x_fit = np.linspace(sp.min(), sp.max(), 200)
        if len(sp) < 3:
            continue
        ax.scatter(sp, mn, color=color, alpha=ALPHA, s=5, label=group)
        try:
            params, _ = curve_fit(fit_fn, sp, mn, p0=p0, maxfev=1000)
            ax.plot(x_fit, fit_fn(x_fit, *params), color=color)
            r, _ = r_coef(sp, mn)
            ax.text(0.95, 0.50, f"{r_label} = {r:.2f}", transform=ax.transAxes,
                    ha="right", va="top", fontsize=8)
        except RuntimeError:
            pass
        ax.set_ylabel(ylabel)
        ax.set_xlabel("Average speed (mm/s)")
    ax_swing.legend()


fig.tight_layout()
plt.show()

In [ ]:
# Stride period (time for all legs to complete their movement cycles) vs speed

fig, axs = plt.subplots(NUM_GROUPS, figsize=(4, 3 * NUM_GROUPS))
for ax, (group, data) in zip(axs, group_data.items()):
    sp, pm = data["period_speeds"], data["period_means"]
    if len(sp) < 3:
        continue
    # Remove extreme outliers before fitting
    mask = pm < np.percentile(pm, 99)
    sp, pm = sp[mask], pm[mask]
    x_fit = np.linspace(sp.min(), sp.max(), 200)
    color = GROUPS[group]["color"]
    ax.scatter(sp, pm, color=color, alpha=ALPHA, s=5, label=group)
    try:
        params, _ = curve_fit(power_law, sp, pm, p0=[500, -0.5], maxfev=1000)
        ax.plot(x_fit, power_law(x_fit, *params), color=color)
        r, _ = spearmanr(sp, pm)
        ax.text(0.95, 0.50, f"Spearman r = {r:.2f}", transform=ax.transAxes,
                ha="right", va="top", fontsize=8)
        ax.set_xlabel("Average speed (mm/s)")
        ax.set_ylabel("Stride period (ms)")
        ax.legend()
    except RuntimeError:
        pass
fig.tight_layout()
# fig.savefig("figures/stride_period.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Step length vs speed


fig, axs = plt.subplots(NUM_GROUPS, figsize=(4, 3 * NUM_GROUPS))
for ax, (group, data) in zip(axs, group_data.items()):
    sp, sl = data["steplen_speeds"], data["steplen_means"]
    x_fit = np.linspace(sp.min(), sp.max(), 200)
    if len(sp) < 3:
        continue
    color = GROUPS[group]["color"]
    ax.scatter(sp, sl, color=color, alpha=ALPHA, s=5, label=group)
    try:
        params, _ = curve_fit(linear, sp, sl, p0=[0.1, 0.5], maxfev=1000)
        ax.plot(x_fit, linear(x_fit, *params), color=color)
        r, _ = spearmanr(sp, sl)
        ax.text(0.95, 0.50, f"Pearson r = {r:.2f}", transform=ax.transAxes,
                ha="right", va="top", fontsize=8)
        ax.set_xlabel("Average speed (mm/s)")
        ax.set_ylabel("Step length (mm)")
        ax.legend()
    except RuntimeError:
        pass


fig.tight_layout()
# fig.savefig("figures/step_length.pdf", bbox_inches="tight")
plt.show()

## Footprint Alignment

In [ ]:
fig, axs = plt.subplots(NUM_GROUPS, figsize=(4, 3 * NUM_GROUPS))
for ax, (group, data) in zip(axs, group_data.items()):
    color = GROUPS[group]["color"]
    ax.scatter(data["fp_speeds"], data["fp_means"],
               color=color, alpha=ALPHA, s=10, label=group)
    ax.set_xlabel("Average speed (mm/s)")
    ax.set_ylabel("Footprint alignment std dev (mm)")
    ax.legend()

fig.tight_layout()
# fig.savefig("figures/footprint_alignment.pdf", bbox_inches="tight")
plt.show()

## Walking Bout Duration

In [ ]:
fig, ax = plt.subplots(figsize=FIG_SIZE)

all_lengths = [(group, data["bout_lengths"]) for group, data in group_data.items()]
max_len = max(l.max() for _, l in all_lengths if len(l))
bins = np.linspace(0, max_len, 30)

for group, lengths in all_lengths:
    hist, edges = np.histogram(lengths, bins=bins, density=True)
    hist = gaussian_filter1d(hist.astype(float), sigma=0.8)
    centers = (edges[:-1] + edges[1:]) / 2
    ax.plot(centers, hist, color=GROUPS[group]["color"], label=group)

ax.set_xlabel("Walking bout duration (s)")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
# fig.savefig("figures/walking_bout_duration.pdf", bbox_inches="tight")
plt.show()